# COS30082 — Approach 2 V3B: Conditional Multi-Task + **Synthetic Pairwise Feature Recombination**

**Standalone training notebook — run from your team's repository (`approaches/` folder).** This is a DPD-Small *research adaptation* inspired by Pairwise Feature Learning, **not** a literal reproduction of Chai et al. (2023).

**Change versus the earlier V3 notebook:** Instead of mixing disease features from two images with the **same disease**, we synthesize **new crop–disease combinations**: crop features from image *i* + disease features from a *different* image *j*. We preferentially choose a `(crop_i, disease_j)` composition that is **absent from the real training split**. Both concepts must be present in training. The model sees the generated feature and learns its known *component* labels; no synthetic image or unseen image is required.

| Experiment | Crop-conditioned disease prediction | Synthetic new pair feature training |
|---|---|---|
| `control` | No | No |
| `exp1_conditional` | Yes | No |
| `exp2_pairwise` | No | Yes |
| `exp3_combined` | Yes | Yes |

**Strict separation:** Development uses only the original 8,000 training-image records; 10 pair categories are withheld *as real images* for OOD development. The dev synthetic sampler **also excludes those 10 held-out pair labels**. Official seen/unseen test **images and per-image labels are never used as training examples, development targets, model-selection labels or calibration data** (the team-provided candidate-pair mapping reads test metadata solely to define the fixed output label space). The team-standard candidate list (`get_pairs()`) is used **only to score the 333 output pairs**, not to choose synthetic training pairs.

**Model selection:** train-only development `macro-held-out H` (seen validation Top-1 vs mean of 10 held-out pair accuracies), tie-break with ordinary micro H. The official report still uses the team's **micro** seen/unseen H. The selected dev epoch is **locked**, then retrained from scratch with the original team's 90/10 seen split covering all 323 seen pairs. Only the locked final checkpoint may be evaluated on official tests.

> **Caveat:** Generated combinations can be biologically implausible because no external crop–disease compatibility knowledge is assumed. Their labels are *compositional training targets* rather than observations of genuine plant disease images. The ablation must show whether this helps or hurts.

## 1. Setup — edit the flags here

In [ ]:
from __future__ import annotations
import csv
import json
import os
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import timm
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

# The notebook can run from the project root OR from project/approaches/.
for root in (Path.cwd(), Path.cwd().parent):
    if (root / "common" / "dataset.py").is_file():
        PROJECT_ROOT = root.resolve()
        sys.path.insert(0, str(PROJECT_ROOT))
        break
else:
    raise RuntimeError("Put this notebook inside project/approaches and open Jupyter from the project root.")

from common import config
from common.dataset import PlantDataset, get_transforms, get_pairs, get_pair_names, load_split, _split_train_val
from common.evaluate import evaluate

# ========== EDIT THESE ONLY ==========
DEBUG = True                      # First do a smoke test; switch to False for formal research.
RUN_DEVELOPMENT = True            # Four controlled experiments.
RUN_FINAL_RETRAIN = False         # After FULL development, set True; RUN_DEVELOPMENT=False.
RUN_OFFICIAL_TEST = False         # Only after final retraining and inspection; one official test.
DEVELOPMENT_RUN_DIR = None        # REQUIRED for FINAL, e.g. "results/approach2_v3b/full_2026..."
FINAL_CHECKPOINT = None           # REQUIRED for OFFICIAL, e.g. "checkpoints/approach2_v3b/final_xxx/locked_final.pt"

EXPERIMENT = "all"                # "all", "control", "exp1_conditional", "exp2_pairwise", "exp3_combined"
BATCH_SIZE = 16 if DEBUG else 64
EPOCHS = 1 if DEBUG else 20
NUM_WORKERS = 0 if os.name == "nt" else 4  # Ubuntu/Docker with --shm-size=8g: 4
LEARNING_RATE_BACKBONE = 1e-5
LEARNING_RATE_HEAD = 1e-4
WEIGHT_DECAY = 1e-4
SEED = 42
CONCEPT_DIM = 512
CONTEXT_DIM = 64
CONTEXT_SCALE = 0.25
AUX_LOSS_WEIGHT = 0.25
SYNTH_MAX_WEIGHT = 0.5
SYNTH_WARMUP_EPOCHS = 0
SYNTH_RAMP_EPOCHS = 3
LABEL_SMOOTHING = 0.1

DEV_RESULTS_ROOT = config.RESULTS_DIR / "approach2_v3b"
CHECKPOINT_ROOT = config.CHECKPOINT_DIR / "approach2_v3b"
DEVICE = config.DEVICE
AMP = DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()

EXPERIMENTS = {
    "control": {"conditional": False, "synthetic": False},
    "exp1_conditional": {"conditional": True, "synthetic": False},
    "exp2_pairwise": {"conditional": False, "synthetic": True},
    "exp3_combined": {"conditional": True, "synthetic": True},
}

# Fixed V2 train-only pair-held-out split: do not change to select a favorable result.
HELDOUT_ROWS = (255, 65, 296, 32, 241, 192, 299, 83, 321, 262)
EXPECTED_HELDOUT_NAMES = (
    "soybean_rust", "cassava_bacterial_blight", "tomato_leaf_blight",
    "basil_healthy", "paddy_leaf_blight", "paddy_healthy",
    "tomato_mosaic_virus", "cherry_healthy", "zucchini_powdery_mildew",
    "sugarcane_brown_spot",
)

assert EXPERIMENT in ("all", *EXPERIMENTS)
assert EPOCHS >= 1 and BATCH_SIZE >= 2
config.set_seed(SEED)
print("Project:", PROJECT_ROOT)
print("Device:", DEVICE, "BF16:", AMP, "Workers:", NUM_WORKERS)
print("DEBUG:", DEBUG, "epochs:", EPOCHS, "batch:", BATCH_SIZE)

## 2. DPD-Small data and leakage checks

In [ ]:
# Deliberately preserve the team common/dataset.py transforms, pair mapping and loader format.
train_frame_full = load_split(config.TRAIN_CSV)
assert len(train_frame_full) == 8000, f"Expected 8000 train records, found {len(train_frame_full)}"

PAIRS_CPU = get_pairs()
PAIR_NAMES = get_pair_names()
assert PAIRS_CPU.shape == (333, 2)
assert all(PAIR_NAMES[idx] == EXPECTED_HELDOUT_NAMES[i]
           for i, idx in enumerate(HELDOUT_ROWS)), "Team pair table differs from the known V2 split"
PAIRS = PAIRS_CPU.to(DEVICE)

# Do not load official test image records here. The shared candidate pair mapping reads
# metadata (for fixed candidate labels) per team evaluation interface, not image pixels.

def make_development_frames():
    is_heldout = train_frame_full["pair_row"].isin(HELDOUT_ROWS)
    heldout = train_frame_full.loc[is_heldout].copy()
    remainder = train_frame_full.loc[~is_heldout].copy()
    seen_val = remainder.groupby("pair_row").sample(frac=0.1, random_state=SEED)
    train_dev = remainder.drop(seen_val.index)
    assert (len(train_dev), len(seen_val), len(heldout)) == (6871, 737, 392)
    for concept in ["crop", "disease"]:
        assert set(heldout[concept]) <= set(train_dev[concept]), f"Missing {concept} in training!"
    assert set(heldout.pair_row).isdisjoint(set(train_dev.pair_row))
    assert not set(train_dev.image) & set(heldout.image)
    assert not set(seen_val.image) & set(heldout.image)
    return train_dev, seen_val, heldout


def make_loader(frame, transform, batch_size=BATCH_SIZE, shuffle=False):
    return DataLoader(
        PlantDataset(frame, transform, norm="imagenet"),
        batch_size=batch_size, shuffle=shuffle,
        num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
        persistent_workers=NUM_WORKERS > 0,
        generator=torch.Generator().manual_seed(SEED),
    )


def make_real_pair_mask(frame):
    # The synthetic sampler ONLY consults labels from its own real training frame.
    mask = torch.zeros((config.NUM_CROPS, config.NUM_DISEASES), dtype=torch.bool)
    crops = torch.as_tensor(frame["crop"].to_numpy(), dtype=torch.long)
    diseases = torch.as_tensor(frame["disease"].to_numpy(), dtype=torch.long)
    mask[crops, diseases] = True
    return mask.to(DEVICE)


def make_forbidden_dev_mask():
    # Extra protection: do not synthesize the ten dev-heldout pair labels.
    mask = torch.zeros((config.NUM_CROPS, config.NUM_DISEASES), dtype=torch.bool)
    for row in HELDOUT_ROWS:
        crop, disease = PAIRS_CPU[row].tolist()
        mask[crop, disease] = True
    return mask.to(DEVICE)

print("Training images:", len(train_frame_full))
print("Candidate pairs:", len(PAIRS_CPU), "; dev heldout pairs:", len(HELDOUT_ROWS))

## 3. Architecture: residual composition + optional soft crop conditioning

The **same `compose(crop_feature, disease_feature)` module** is used for (a) real single-image features and (b) synthesized cross-image pairs. A separate auxiliary disease head keeps disease-specific supervision. No ground-truth crop is used at inference: the model evaluates all candidate crop conditions and scores the 333 allowed pairs.

In [ ]:
class PairwiseCompositionalNet(nn.Module):
    def __init__(self, conditional: bool, pretrained: bool = True):
        super().__init__()
        self.conditional = bool(conditional)
        self.backbone = timm.create_model("resnet50", pretrained=pretrained, num_classes=0)
        feature_size = self.backbone.num_features
        self.crop_encoder = nn.Sequential(nn.Linear(feature_size, CONCEPT_DIM), nn.ReLU(), nn.LayerNorm(CONCEPT_DIM))
        self.disease_encoder = nn.Sequential(nn.Linear(feature_size, CONCEPT_DIM), nn.ReLU(), nn.LayerNorm(CONCEPT_DIM))

        self.crop_aux_head = nn.Linear(CONCEPT_DIM, config.NUM_CROPS)
        self.disease_aux_head = nn.Linear(CONCEPT_DIM, config.NUM_DISEASES)

        # Shared for REAL (crop_i, disease_i) and SYNTHETIC (crop_i, disease_j).
        self.pair_residual = nn.Sequential(
            nn.Linear(2 * CONCEPT_DIM, CONCEPT_DIM), nn.GELU(),
            nn.Linear(CONCEPT_DIM, CONCEPT_DIM),
        )
        self.pair_norm = nn.LayerNorm(CONCEPT_DIM)
        self.pair_crop_head = nn.Linear(CONCEPT_DIM, config.NUM_CROPS)
        self.pair_disease_head = nn.Linear(CONCEPT_DIM, config.NUM_DISEASES)

        if self.conditional:
            self.crop_context = nn.Linear(CONCEPT_DIM, CONTEXT_DIM)
            self.crop_embedding = nn.Embedding(config.NUM_CROPS, CONTEXT_DIM)
            self.context_gate = nn.Linear(CONTEXT_DIM, CONCEPT_DIM)
            # The conditional residual starts neutral, reducing instability early in training.
            nn.init.zeros_(self.context_gate.weight)
            nn.init.zeros_(self.context_gate.bias)

    def encode(self, images):
        shared = self.backbone(images)
        return self.crop_encoder(shared), self.disease_encoder(shared)

    def compose(self, crop_z, disease_z):
        if crop_z.shape != disease_z.shape:
            raise ValueError("Crop/Disease concept feature tensors must have matching shape")
        residual = torch.tanh(self.pair_residual(torch.cat((crop_z, disease_z), dim=-1)))
        return self.pair_norm(crop_z + disease_z + 0.25 * residual)

    def conditioned_disease_logits(self, pair_z, crop_z, crop_ids):
        """Real/synthetic batch, given a known crop label FOR TRAINING ONLY."""
        if not self.conditional:
            return self.pair_disease_head(pair_z)
        context = self.crop_context(crop_z) * self.crop_embedding(crop_ids)
        gate = torch.tanh(self.context_gate(context))
        return self.pair_disease_head(pair_z * (1 + CONTEXT_SCALE * gate))

    def real_losses(self, images, crop_ids, disease_ids):
        crop_z, disease_z = self.encode(images)
        real_pair_z = self.compose(crop_z, disease_z)
        crop_logits = self.pair_crop_head(real_pair_z)
        disease_logits = self.conditioned_disease_logits(real_pair_z, crop_z, crop_ids)
        main = (F.cross_entropy(crop_logits, crop_ids, label_smoothing=LABEL_SMOOTHING)
                + F.cross_entropy(disease_logits, disease_ids, label_smoothing=LABEL_SMOOTHING))
        auxiliary = (
            F.cross_entropy(self.crop_aux_head(crop_z), crop_ids, label_smoothing=LABEL_SMOOTHING)
            + F.cross_entropy(self.disease_aux_head(disease_z), disease_ids,
                              label_smoothing=LABEL_SMOOTHING)
        )
        return crop_z, disease_z, main, auxiliary

    def synthetic_loss(self, crop_z, disease_z, crop_ids, disease_ids, anchor, partner):
        # Crop from image i; disease from image j; labels are (crop_i, disease_j).
        synthetic_z = self.compose(crop_z[anchor], disease_z[partner])
        target_crop = crop_ids[anchor]
        target_disease = disease_ids[partner]
        logits_crop = self.pair_crop_head(synthetic_z)
        logits_disease = self.conditioned_disease_logits(synthetic_z, crop_z[anchor], target_crop)
        return (F.cross_entropy(logits_crop, target_crop, label_smoothing=LABEL_SMOOTHING)
                + F.cross_entropy(logits_disease, target_disease, label_smoothing=LABEL_SMOOTHING))

    def predict(self, images, pairs):
        """Inference: output [B,333] scores WITHOUT ground-truth crop/disease."""
        crop_z, disease_z = self.encode(images)
        pair_z = self.compose(crop_z, disease_z)
        crop_log = F.log_softmax(self.pair_crop_head(pair_z).float(), dim=1)
        if self.conditional:
            # [B,55,CONCEPT_DIM], then [B,55,175]
            all_crop_ids = torch.arange(config.NUM_CROPS, device=images.device)
            context = self.crop_context(crop_z)[:, None, :] * self.crop_embedding(all_crop_ids)[None, :, :]
            gate = torch.tanh(self.context_gate(context))
            z_per_crop = pair_z[:, None, :] * (1 + CONTEXT_SCALE * gate)
            disease_log_by_crop = F.log_softmax(self.pair_disease_head(z_per_crop).float(), dim=-1)
            pair_disease_log = disease_log_by_crop[:, pairs[:, 0], pairs[:, 1]]
            disease_marginal = torch.logsumexp(crop_log[:, :, None] + disease_log_by_crop, dim=1)
        else:
            disease_marginal = F.log_softmax(self.pair_disease_head(pair_z).float(), dim=1)
            pair_disease_log = disease_marginal[:, pairs[:, 1]]
        scores = crop_log[:, pairs[:, 0]] + pair_disease_log
        return scores, crop_log, disease_marginal

## 4. Novel-pair sampler, training and diagnostics

Pairs are formed from **two different training images**. The requested target `(crop_i, disease_j)` must be **absent from the real training split**. For dev, it also cannot be one of the 10 held-out validation pair labels. Synthetic samples provide concept labels only, not real unseen-image pixels. We track `novel_anchor_pct` to prove generation actually occurred.

In [ ]:
def choose_novel_partners(crop_ids, disease_ids, real_pair_mask, forbidden_pair_mask=None):
    """One uniformly randomized eligible partner j per eligible anchor i."""
    b = len(crop_ids)
    forbidden = real_pair_mask[crop_ids[:, None], disease_ids[None, :]]
    if forbidden_pair_mask is not None:
        forbidden = forbidden | forbidden_pair_mask[crop_ids[:, None], disease_ids[None, :]]
    indices = torch.arange(b, device=crop_ids.device)
    eligible = (~forbidden) & (indices[:, None] != indices[None, :])
    eligible &= (disease_ids[:, None] != disease_ids[None, :])
    anchors = eligible.any(dim=1).nonzero(as_tuple=True)[0]
    if anchors.numel() == 0:
        return anchors, anchors
    samples = torch.rand((len(anchors), b), device=crop_ids.device)
    partners = samples.masked_fill(~eligible[anchors], -1.0).argmax(dim=1)
    # Every synthetic label is absent from the real training pairs and dev-heldout pairs.
    assert not real_pair_mask[crop_ids[anchors], disease_ids[partners]].any()
    if forbidden_pair_mask is not None:
        assert not forbidden_pair_mask[crop_ids[anchors], disease_ids[partners]].any()
    return anchors, partners


def synth_weight_for_epoch(epoch):
    if epoch <= SYNTH_WARMUP_EPOCHS:
        return 0.0
    fraction = min(1.0, (epoch - SYNTH_WARMUP_EPOCHS) / max(1, SYNTH_RAMP_EPOCHS))
    return SYNTH_MAX_WEIGHT * fraction


def train_one_epoch(model, loader, optimizer, device, synthetic, real_pair_mask,
                    forbidden_pair_mask, epoch):
    model.train()
    total, main_total, auxiliary_total, synth_total, composed_count = 0., 0., 0., 0., 0
    synth_weight = synth_weight_for_epoch(epoch) if synthetic else 0.0
    for images, crops, diseases, _ in loader:
        images = images.to(device, non_blocking=True)
        crops = crops.to(device, non_blocking=True)
        diseases = diseases.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=AMP):
            crop_z, disease_z, main_loss, aux_loss = model.real_losses(images, crops, diseases)
            synth_loss = main_loss.new_zeros(())
            if synth_weight > 0:
                anchors, partners = choose_novel_partners(crops, diseases, real_pair_mask, forbidden_pair_mask)
                composed_count += anchors.numel()
                if anchors.numel():
                    synth_loss = model.synthetic_loss(crop_z, disease_z, crops, diseases, anchors, partners)
            loss = main_loss + AUX_LOSS_WEIGHT * aux_loss + synth_weight * synth_loss
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        n = len(crops)
        total += float(loss.detach()) * n
        main_total += float(main_loss.detach()) * n
        auxiliary_total += float(aux_loss.detach()) * n
        synth_total += float(synth_loss.detach()) * n
    denom = len(loader.dataset)
    return {
        "loss": total/denom, "real_main_loss": main_total/denom,
        "real_aux_loss": auxiliary_total/denom, "synthetic_loss": synth_total/denom,
        "synthetic_weight": synth_weight, "novel_anchor_pct": 100.0*composed_count/denom,
    }


@torch.inference_mode()
def validation_metrics(model, loader, pairs, train_pair_mask, device):
    model.eval()
    counters = torch.zeros(6, dtype=torch.long, device=device)
    totals_by_pair = torch.zeros(len(pairs), dtype=torch.long, device=device)
    correct_by_pair = torch.zeros(len(pairs), dtype=torch.long, device=device)
    for images, crops, diseases, targets in loader:
        images, crops, diseases, targets = [x.to(device, non_blocking=True)
                                           for x in (images, crops, diseases, targets)]
        scores, crop_log, disease_log = model.predict(images, pairs)
        predictions = scores.topk(5, dim=1).indices
        pred_top1 = predictions[:, 0]
        correct = pred_top1 == targets
        counters += torch.stack([
            correct.sum(), (predictions == targets[:, None]).any(dim=1).sum(),
            (crop_log.argmax(dim=1) == crops).sum(),
            (disease_log.argmax(dim=1) == diseases).sum(),
            train_pair_mask[pred_top1].sum(),
            (pairs[pred_top1, 0] == crops).sum(),
        ])
        totals_by_pair += torch.bincount(targets, minlength=len(pairs))
        correct_by_pair += torch.bincount(targets[correct], minlength=len(pairs))
    n = len(loader.dataset)
    keys = ["pair_top1", "pair_top5", "crop_head_top1", "disease_head_top1",
            "predicted_real_training_pair_pct", "top_pair_crop_top1"]
    output = {key: 100.0*value/n for key, value in zip(keys, counters.cpu().tolist())}
    present = totals_by_pair > 0
    output["macro_pair_top1"] = 100.0 * float((correct_by_pair[present] / totals_by_pair[present]).float().mean())
    rows = [{"pair_row": row, "pair_name": PAIR_NAMES[row], "n_images": size,
             "n_correct": correct, "pair_top1": (100.0*correct/size)}
            for row,(size, correct) in enumerate(zip(totals_by_pair.cpu().tolist(),
                                                       correct_by_pair.cpu().tolist())) if size]
    return output, rows


def harmonic(a,b):
    return 2*a*b/(a+b) if a+b else 0.0


def selection_key(row):
    # Equal weight to each of the 10 heldout pair categories to avoid "paddy_healthy" dominance.
    return (row["macro_harmonic_mean"], row["micro_harmonic_mean"],
            row["heldout_macro_pair_top1"], row["seen_pair_top1"])

## 5. Train four V3B ablations (development ONLY)

In [ ]:
def write_csv(path, rows):
    if not rows:
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def write_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, default=str)


def save_checkpoint_atomic(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(".tmp")
    torch.save(payload, temporary)
    os.replace(temporary, path)


def make_optimizer(model):
    other_params = [p for name,p in model.named_parameters() if not name.startswith("backbone.")]
    return torch.optim.AdamW([
        {"params": model.backbone.parameters(), "lr": LEARNING_RATE_BACKBONE},
        {"params": other_params, "lr": LEARNING_RATE_HEAD},
    ], weight_decay=WEIGHT_DECAY)


def development_run():
    train, seen, heldout = make_development_frames()
    if DEBUG:
        train = train.sample(n=min(len(train),128), random_state=SEED)
        seen = seen.sample(n=min(len(seen),64), random_state=SEED)
        heldout = heldout.sample(n=min(len(heldout),80), random_state=SEED)
    real_pair_mask = make_real_pair_mask(train)
    forbidden_dev_mask = make_forbidden_dev_mask()
    pair_mask_for_eval = torch.zeros(len(PAIRS), dtype=torch.bool, device=DEVICE)
    pair_mask_for_eval[torch.as_tensor(train["pair_row"].unique(), dtype=torch.long, device=DEVICE)] = True
    train_transform, test_transform = get_transforms("imagenet")
    seen_loader = make_loader(seen, test_transform)
    heldout_loader = make_loader(heldout, test_transform)
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    name = "debug" if DEBUG else "full"
    results_dir = DEV_RESULTS_ROOT / f"{name}_{stamp}"
    checkpoints_dir = CHECKPOINT_ROOT / f"{name}_{stamp}"
    results_dir.mkdir(parents=True, exist_ok=True)
    checkpoints_dir.mkdir(parents=True, exist_ok=True)
    write_json(results_dir / "split_and_settings.json", {
        "debug": DEBUG, "train_count": len(train), "seen_validation_count": len(seen),
        "heldout_validation_count": len(heldout), "heldout_pairs": list(HELDOUT_ROWS),
        "seed": SEED, "batch_size": BATCH_SIZE, "epochs": EPOCHS,
        "selection": "macro_harmonic_mean, then micro_harmonic_mean",
        "synth_policy": "cross-image crop_i+disease_j, absent in real training pairs; dev-heldout pair labels forbidden",
        "learning_rate_backbone": LEARNING_RATE_BACKBONE,
        "learning_rate_head": LEARNING_RATE_HEAD,
        "synth_max_weight": SYNTH_MAX_WEIGHT,
        "synth_warmup_epochs": SYNTH_WARMUP_EPOCHS,
        "synth_ramp_epochs": SYNTH_RAMP_EPOCHS,
        "checkpoint_dir": str(checkpoints_dir.resolve()),
    })
    selected_names = list(EXPERIMENTS) if EXPERIMENT == "all" else [EXPERIMENT]
    experiment_bests = []
    print(f"Development: {len(train)} train / {len(seen)} seen val / {len(heldout)} held-out val")
    print("Results:", results_dir, "\nCheckpoints:", checkpoints_dir, flush=True)
    for experiment_name in selected_names:
        flags = EXPERIMENTS[experiment_name]
        config.set_seed(SEED)
        model = PairwiseCompositionalNet(conditional=flags["conditional"], pretrained=True).to(DEVICE)
        optimizer = make_optimizer(model)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
        train_loader = make_loader(train, train_transform, shuffle=True)
        best = None
        history = []
        for epoch in range(1, EPOCHS+1):
            train_metrics = train_one_epoch(model, train_loader, optimizer, DEVICE,
                                            flags["synthetic"], real_pair_mask, forbidden_dev_mask, epoch)
            scheduler.step()
            seen_metrics, _ = validation_metrics(model, seen_loader, PAIRS, pair_mask_for_eval, DEVICE)
            held_metrics, held_rows = validation_metrics(model, heldout_loader, PAIRS, pair_mask_for_eval, DEVICE)
            row = {
                "experiment": experiment_name, "epoch": epoch, **train_metrics,
                **{f"seen_{k}":v for k,v in seen_metrics.items()},
                **{f"heldout_{k}":v for k,v in held_metrics.items()},
                "micro_harmonic_mean": harmonic(seen_metrics["pair_top1"],held_metrics["pair_top1"]),
                "macro_harmonic_mean": harmonic(seen_metrics["pair_top1"],held_metrics["macro_pair_top1"]),
            }
            history.append(row)
            write_csv(results_dir / f"{experiment_name}_history.csv", history)
            print(f"{experiment_name:>18} {epoch:02d}/{EPOCHS}: "
                  f"seen={seen_metrics['pair_top1']:.2f}% "
                  f"heldout={held_metrics['pair_top1']:.2f}% "
                  f"heldout_macro={held_metrics['macro_pair_top1']:.2f}% "
                  f"H_macro={row['macro_harmonic_mean']:.2f}% "
                  f"novel_anchor={train_metrics['novel_anchor_pct']:.1f}%", flush=True)
            if best is None or selection_key(row) > selection_key(best):
                best = dict(row)
                best["checkpoint"] = str((checkpoints_dir / f"{experiment_name}_best.pt").resolve())
                write_csv(results_dir / f"{experiment_name}_best_heldout_pairs.csv", held_rows)
                if not DEBUG:
                    save_checkpoint_atomic(Path(best["checkpoint"]), {
                        "state_dict": model.state_dict(), "experiment": experiment_name,
                        "epoch": epoch, "selection_key": list(selection_key(row)),
                        "training_source": "train.csv only", "phase": "development",
                    })
        if flags["synthetic"] and EPOCHS > SYNTH_WARMUP_EPOCHS and not any(
                x["novel_anchor_pct"] > 0 for x in history):
            raise RuntimeError(f"{experiment_name} did not synthesize ANY novel pair: inspect sampler/split")
        experiment_bests.append(best)
        write_csv(results_dir / "validation_summary.csv", sorted(experiment_bests,
                                                        key=selection_key, reverse=True))
        del model, optimizer, scheduler, train_loader
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    winner = max(experiment_bests, key=selection_key)
    write_json(results_dir / "selection.json", {
        "winner": winner["experiment"], "selected_dev_epoch": int(winner["epoch"]),
        "winner_metrics": winner, "all_candidates": experiment_bests,
        "selection_policy": "max macro harmonic mean; then micro harmonic mean, then heldout macro; then seen",
        "debug": DEBUG, "checkpoint_root": str(checkpoints_dir.resolve()),
        "note": "DO NOT use debug run or dev checkpoint for official evaluation. Retrain on all original seen pairs.",
    })
    print("DEV WINNER:", winner["experiment"], "at epoch", winner["epoch"],
          "macro H:",round(winner["macro_harmonic_mean"],2))
    print("SAVE THIS DEVELOPMENT_RUN_DIR:", results_dir.resolve())
    return results_dir

## 6. Execute development (set `DEBUG=True` for a quick dry-run)

In [ ]:
if RUN_DEVELOPMENT:
    development_run_dir = development_run()
else:
    print("Development skipped. Set DEVELOPMENT_RUN_DIR to an existing FULL run for final retraining.")

## 7. Inspect and decide — read development metrics BEFORE final training

In [ ]:
if RUN_DEVELOPMENT:
    print("Selected configuration (DEVELOPMENT ONLY):")
    print((development_run_dir / "selection.json").read_text(encoding="utf-8")[:3500])
    display(pd.read_csv(development_run_dir / "validation_summary.csv")[
        ["experiment", "epoch", "seen_pair_top1", "heldout_pair_top1",
         "heldout_macro_pair_top1", "heldout_disease_head_top1",
         "heldout_predicted_real_training_pair_pct", "macro_harmonic_mean",
         "micro_harmonic_mean", "novel_anchor_pct"]
    ].round(2))

## 8. LOCKED final retraining on the original team's 90/10 split

**Not the dev-heldout split.** We rebuild a fresh model using **all 323 original seen pair types** (the standard training 90% split). The dev-selected experiment and epoch are **fixed in advance**. We do **not** choose a checkpoint based on official tests or re-select using only the seen validation metric. The original team's *10%-per-pair* seen validation examples are kept for diagnostics, as in the baselines.

This means final epoch count may be less than 20 if train-only OOD validation selected an early epoch. State it openly in your report. If you want a strict 20-epoch comparison, predeclare that alternative in a separate controlled study; don't switch policies after seeing official scores.

In [ ]:
def get_explicit_development_dir():
    if DEVELOPMENT_RUN_DIR is None:
        raise ValueError("Set DEVELOPMENT_RUN_DIR in Cell 1 to your FULL development results folder.")
    path = Path(DEVELOPMENT_RUN_DIR)
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    assert path.is_dir(), f"Cannot find development directory: {path}"
    selection = json.loads((path / "selection.json").read_text(encoding="utf-8"))
    assert not selection["debug"], "A DEBUG run cannot select the final configuration."
    return path, selection


def final_retrain():
    if DEBUG:
        raise ValueError("Final retraining requires DEBUG=False")
    dev_path, selection = get_explicit_development_dir()
    exp = selection["winner"]
    epochs_to_train = int(selection["selected_dev_epoch"])
    assert 1 <= epochs_to_train <= 20
    flags = EXPERIMENTS[exp]
    config.set_seed(SEED)
    # EXACT standard team 90/10 split, covering all 323 seen pair types.
    original_train, original_val = _split_train_val(load_split(config.TRAIN_CSV))
    assert len(original_train) + len(original_val) == 8000
    assert original_train["pair_row"].nunique() == 323
    train_tf, val_tf = get_transforms("imagenet")
    train_loader = make_loader(original_train, train_tf, shuffle=True)
    seen_val_loader = make_loader(original_val, val_tf)
    # NO forbidden official-unseen labels: only the original training CSV is used to build the sampler.
    real_mask = make_real_pair_mask(original_train)
    mask_for_eval = torch.zeros(len(PAIRS),dtype=torch.bool,device=DEVICE)
    mask_for_eval[torch.as_tensor(original_train.pair_row.unique(),device=DEVICE)] = True
    final_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    output = CHECKPOINT_ROOT / f"final_{final_id}"
    output.mkdir(parents=True, exist_ok=True)
    print(f"LOCKED: experiment={exp}; epoch={epochs_to_train}; train={len(original_train)}; "
          f"seen_val={len(original_val)}", flush=True)
    model = PairwiseCompositionalNet(flags["conditional"],pretrained=True).to(DEVICE)
    optimizer = make_optimizer(model)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=20)
    history = []
    for epoch in range(1, epochs_to_train+1):
        metrics = train_one_epoch(model, train_loader, optimizer, DEVICE, flags["synthetic"],
                                  real_mask, None, epoch)
        scheduler.step()
        val, _ = validation_metrics(model, seen_val_loader, PAIRS, mask_for_eval, DEVICE)
        row = {"epoch": epoch, **metrics, **{f"seen_val_{k}":v for k,v in val.items()}}
        history.append(row)
        write_csv(output / "locked_final_history.csv", history)
        print(f"Final {epoch}/{epochs_to_train} seen_val={val['pair_top1']:.2f}% "
              f"novel_anchor={metrics['novel_anchor_pct']:.1f}%", flush=True)
    ckpt = output / "locked_final.pt"
    save_checkpoint_atomic(ckpt, {
        "state_dict": model.state_dict(), "experiment": exp,
        "epoch": epochs_to_train, "phase": "locked_final",
        "training_split": "common.dataset._split_train_val(train.csv), all 323 seen pair types",
        "development_selection": str((dev_path / "selection.json").resolve()),
        "seed": SEED, "batch_size": BATCH_SIZE,
    })
    write_json(output / "locked_final_metadata.json", {
        "experiment":exp, "epoch":epochs_to_train, "training_images":len(original_train),
        "seen_validation_images":len(original_val), "checkpoint":str(ckpt.resolve()),
        "development_selection":str((dev_path / "selection.json").resolve()),
        "official_test_evaluated": False,
    })
    print("LOCKED FINAL CHECKPOINT:",ckpt.resolve())
    return ckpt

## 9. Execute final retraining ONLY after FULL development and review

In [ ]:
if RUN_FINAL_RETRAIN:
    final_checkpoint_path = final_retrain()
else:
    print("Final retraining skipped; RUN_FINAL_RETRAIN=False")

## 10. OPTIONAL: official evaluation of the locked FINAL checkpoint (do once)

In [ ]:
def official_evaluate_locked_final():
    if DEBUG:
        raise ValueError("Official evaluation requires DEBUG=False")
    if FINAL_CHECKPOINT is None:
        raise ValueError("Set FINAL_CHECKPOINT in Cell 1 to the saved locked_final.pt before testing")
    ckpt_path = Path(FINAL_CHECKPOINT)
    if not ckpt_path.is_absolute():
        ckpt_path = PROJECT_ROOT / ckpt_path
    saved = torch.load(ckpt_path, map_location="cpu", weights_only=True)
    assert saved.get("phase") == "locked_final", "Refusing to evaluate a DEVELOPMENT checkpoint on official tests"
    exp = saved["experiment"]
    assert exp in EXPERIMENTS
    model = PairwiseCompositionalNet(EXPERIMENTS[exp]["conditional"], pretrained=False).to(DEVICE)
    model.load_state_dict(saved["state_dict"])
    model.eval()
    _, test_transform = get_transforms("imagenet")
    # Official TEST images are read here for evaluation ONLY.
    official_seen = make_loader(load_split(config.TEST_SEEN_CSV),test_transform)
    official_unseen = make_loader(load_split(config.TEST_UNSEEN_CSV),test_transform)
    assert len(official_seen.dataset)==2000 and len(official_unseen.dataset)==1448
    out_dir = DEV_RESULTS_ROOT / f"official_locked_{datetime.now():%Y%m%d_%H%M%S_%f}"
    out_dir.mkdir(parents=True, exist_ok=True)
    # Team-standard evaluate() saves JSON, seen confusion CSV/PNG, harmonic mean, top1/5.
    original_dir = config.RESULTS_DIR
    config.RESULTS_DIR = out_dir
    try:
        hp = {"model":"approach2_v3b", "experiment":exp,
              "final_epoch":int(saved["epoch"]), "seed":SEED,
              "training_split":saved["training_split"],
              "dev_selection":saved["development_selection"],
              "synthetic_new_pairs_from_training_only":EXPERIMENTS[exp]["synthetic"],
              "no_ground_truth_crop_at_inference":True}
        with torch.inference_mode():
            result = evaluate(lambda x: model.predict(x.to(DEVICE), PAIRS)[0],
                              official_seen,official_unseen,
                              model_name=f"approach2_v3b_{exp}",hp=hp,robust=True)
    finally:
        config.RESULTS_DIR = original_dir
    write_json(out_dir / "evaluation_provenance.json", {
        "final_checkpoint":str(ckpt_path.resolve()), "dev_selection":saved["development_selection"],
        "official_test_used_for_selection":False,
        "single_locked_model":True,
    })
    print("OFFICIAL RESULTS:",out_dir.resolve())
    return result

if RUN_OFFICIAL_TEST:
    official_results = official_evaluate_locked_final()
else:
    print("Official tests remain untouched. RUN_OFFICIAL_TEST=False")

## 11. How to run this notebook

1. **School GPU / first time:** put file at `COS30082-Plant-Disease-OOD/approaches/approach2_v3_optionB_pairwise.ipynb`, open through Jupyter in the repository mounted as `/workspace/project`. Run all with **`DEBUG=True`**, `RUN_DEVELOPMENT=True`, the other `RUN_*` flags False. Inspect the output.
2. **Formal ablations:** `DEBUG=False`, `RUN_DEVELOPMENT=True`, `RUN_FINAL_RETRAIN=False`, `RUN_OFFICIAL_TEST=False` and rerun the notebook. Four independent experiments × 20 epochs. Save the printed **full run directory**, e.g. `results/approach2_v3b/full_...`.
3. **Decide first:** review `validation_summary.csv`, `selection.json`, `*_best_heldout_pairs.csv`, `*_history.csv`. If novel-anchor rate remains 0 or only one held-out pair succeeds, **do not rush to official testing**; refine the methodology using train-only validation.
4. **Lock and retrain:** set `DEBUG=False`, `RUN_DEVELOPMENT=False`, `RUN_FINAL_RETRAIN=True`, `RUN_OFFICIAL_TEST=False`, and set `DEVELOPMENT_RUN_DIR` to your **full** development output path. Run all cells; it saves `checkpoints/approach2_v3b/final_.../locked_final.pt`.
5. **One official evaluation:** set `RUN_DEVELOPMENT=False`, `RUN_FINAL_RETRAIN=False`, `RUN_OFFICIAL_TEST=True`, and set `FINAL_CHECKPOINT` to the **locked final** `.pt` path. Run all cells. Outputs are under `results/approach2_v3b/official_locked_...`.

**Report limitations:** Not a paper reproduction, samples are synthesized **features**, and arbitrary recombinations can be biologically impossible. Dev-macro-H chooses the locked epoch; official micro-H follows the team's standard metrics. Do not compare development checkpoints with the baseline's official test results.

**Environment:** On your school RTX 5090 WSL2 Docker setup, use `--gpus all --shm-size=8g` and `NUM_WORKERS=4`. If you see DataLoader shared-memory errors, confirm Docker was launched with the required shared-memory setting instead of silently changing data or image files.